# 13-amaliyot — Hugging Face bilan BERT nozik sozlash (m13)

**Mavzu:** Oldindan o'qitilgan BERT modelini o'zbek sentiment tahliliga fine-tune qilamiz.
**Kuni:** 14-kun (Day 14) · **Juft ma'ruza:** [L13 — Transfer Learning](../lectures/d13_transfer_learning.pdf)
**Quriladigan kapstone moduli:** `m13 FineTunedClassifier` (haqiqiy modul — `consumed_by`: m15 agent, app.py FastAPI).

Bugun L13 da o'rgangan **transfer learning**, **BERT** va **Hugging Face Trainer** ni amalda qo'llaymiz.

**Bugungi maqsadlar (L13 [C] dan):**
1. Fine-tuning BCE yo'qotishini hisoblaymiz (L13 [I1]: $\sigma(2.0)\approx0.880$, BCE$\approx0.128$).
2. `AutoModelForSequenceClassification` + `Trainer` bilan DistilBERT'ni fine-tune qilamiz.
3. Modelni baholaymiz va `m13` ga ulab, `predict`/`predict_proba` bilan ishlatamiz.

> ⚠️ **Korpus litsenziyasi:** `risqaliyevds/uzbek-sentiment-analysis` (MIT ✓). Reyting {4,5}→ijobiy,
> {1,2}→salbiy, 3 tashlanadi. Offline rejimda kichik original uz sentiment korpusi ishlatiladi.
> **Yorliqlar QULFLANGAN:** `ijobiy` / `salbiy` (L2 [I2] bilan bir xil).

## 1. Muhit tekshiruvi

Kutubxonalar, urug' va bayroqlar. `transformers`/`datasets`/GPU bo'lmasa ham notebook uchdan-uchgacha
ishlaydi: offline rejimda **TF-IDF + LogisticRegression** fallback (m02 naqshi) ishlatiladi.

In [ ]:
import os, sys, random, warnings
warnings.filterwarnings("ignore")
import numpy as np

OFFLINE_FALLBACK = True          # mahalliy: og'ir DistilBERT yuklab olishdan qochamiz
random.seed(42); np.random.seed(42)

try:
    import torch
    torch.manual_seed(42)
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

try:
    import transformers          # noqa: F401
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

try:
    import datasets              # noqa: F401
    HAS_DATASETS = True
except ImportError:
    HAS_DATASETS = False

# Kaggle (GPU+internet): OFFLINE_FALLBACK=False -> haqiqiy DistilBERT fine-tune.
# Mahalliy/offline: fallback (LogReg) -> tez, internetsiz, deterministik.
USE_TRANSFORMERS = HAS_TRANSFORMERS and not OFFLINE_FALLBACK

print("torch:", (torch.__version__ if HAS_TORCH else "yo'q"),
      "| transformers:", ("bor" if HAS_TRANSFORMERS else "yo'q"),
      "| datasets:", ("bor" if HAS_DATASETS else "yo'q"))
print("OFFLINE_FALLBACK =", OFFLINE_FALLBACK, "| USE_TRANSFORMERS =", USE_TRANSFORMERS)

for _cand in ["../../capstone/modules", "/kaggle/working/capstone/modules", "capstone/modules"]:
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand); break

CKPT = "d14_checkpoints"
CORPUS_PATH = os.path.join(CKPT, "uz_sentiment_mini.csv")

## 2. Yaxlit natija (avval manzil)

Tugallangan natijani ko'ramiz: korpusda modelni o'qitamiz va bir sharh uchun sentiment (ijobiy/salbiy)
va ehtimolni olamiz.

In [ ]:
import m13_bert_classifier as M13
from m13_bert_classifier import FineTunedClassifier
M13.USE_TRANSFORMERS = USE_TRANSFORMERS      # offline -> False (LogReg fallback)

import csv
def load_sentiment(path):
    """text<TAB>label CSV ni o'qiydi. Yorliqlar: ijobiy / salbiy."""
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f, delimiter="\t"):
            texts.append(row["text"]); labels.append(row["label"])
    return texts, labels

sharhlar, yorliqlar = load_sentiment(CORPUS_PATH)
print("Sharhlar:", len(sharhlar),
      "| ijobiy:", yorliqlar.count("ijobiy"), "salbiy:", yorliqlar.count("salbiy"))

demo = FineTunedClassifier()
demo.fit(sharhlar, yorliqlar)                # Kaggle: DistilBERT+Trainer; offline: LogReg
namuna = "mahsulot juda sifatli va arzon"
print(repr(namuna), "->", demo.predict(namuna))
print("ehtimol:", {k: round(v, 2) for k, v in demo.predict_proba(namuna).items()})

## 3. Tayyor kod bloki — periferiya (PRIMM)

Bu yerda **to'liq berilgan** kod: sentiment korpusini yuklash va **binarizatsiya** (reyting → ijobiy/salbiy),
hamda Hugging Face tokenizatsiya oqimi (Kaggle yo'li uchun).

### Bashorat qiling
Reyting 1–5 yulduz. Sizningcha, 3 yulduzli sharh qaysi sinfga tushadi? (Quyidagi `binarize` nima qaytaradi?)

In [ ]:
def binarize(rating):
    """Uzum reyting -> sentiment yorliq. 3 (neytral) tashlanadi."""
    if rating >= 4:
        return "ijobiy"
    if rating <= 2:
        return "salbiy"
    return None        # 3 yulduz: noaniq -> tashlanadi

for r in [5, 4, 3, 2, 1]:
    print(r, "yulduz ->", binarize(r))

# Kaggle (real korpus): from datasets import load_dataset
#   ds = load_dataset("risqaliyevds/uzbek-sentiment-analysis")
#   texts/labels = binarize(...) bilan tayyorlanadi
# Hugging Face tokenizatsiya (Kaggle):
#   from transformers import AutoTokenizer, DataCollatorWithPadding
#   tok = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")
#   enc = tok(texts, truncation=True)   ; collator = DataCollatorWithPadding(tok)
# Offline: korpus allaqachon ijobiy/salbiy yorliqli (d14_checkpoints)
print("\nNamuna:", repr(sharhlar[0]), "| yorliq:", yorliqlar[0])

### Tekshiring
1. `binarize(3)` nima qaytaradi va nega u tashlab yuboriladi?
2. Nega neytral (3 yulduz) sharhlarni olib tashlaymiz? (Ikkilik klassifikatsiya aniqroq bo'ladi.)
3. `AutoTokenizer` qaysi modeldan yuklanishi kerak — nega model bilan bir xil `name`?

### O'zgartiring
`binarize` chegarasini o'zgartiring (masalan, faqat 5→ijobiy, 1→salbiy). Korpus balansiga ta'sirini ko'ring.

### ✅ Checkpoint A
Korpusni qaytadan yuklaydi — oldingi kataklar ishlamagan bo'lsa, shu yerdan ishchi holatga qaytamiz.

In [ ]:
if OFFLINE_FALLBACK or "sharhlar" not in dir():
    sharhlar, yorliqlar = load_sentiment(CORPUS_PATH)
print("Checkpoint A: %d sharh tayyor (ijobiy=%d, salbiy=%d)." % (
      len(sharhlar), yorliqlar.count("ijobiy"), yorliqlar.count("salbiy")))

## 4. Asosiy mavzu — so'nuvchi tayanch

Avval L13 [I1] dagi BCE yo'qotishini tekshiramiz, so'ng fine-tuning konfiguratsiyasini birgalikda
quramiz, oxirida `m13` ni o'z korpusimizda ishlatamiz.

### 4A. Namuna (men qilaman): fine-tuning BCE yo'qotishi

L13 [I1]: ishonchli to'g'ri bashorat (logit $z=2.0$, yorliq ijobiy) past yo'qotish beradi. **Toza-torch**
(HF shart emas).

In [ ]:
import torch
import torch.nn as nn

z = torch.tensor([2.0])      # model logiti
y = torch.tensor([1.0])      # ijobiy yorliq (y=1)
p = torch.sigmoid(z)
loss = nn.BCEWithLogitsLoss()(z, y)
print("sigma(2.0) =", round(float(p), 3), "| BCE =", round(float(loss), 3))
assert abs(float(p) - 0.880) < 1e-2
assert abs(float(loss) - 0.128) < 1e-2   # Ma'ruza L13 [I1]-slayd bilan solishtiring
print("To'g'ri! sigma(2.0)=0.880, BCE=0.128 (L13 [I1] bilan mos).")

### 4B. Birgalikda (biz qilamiz): Trainer giperparametrlari

L13 [K4] dagi fine-tuning giperparametrlarini to'ldiring. (Haqiqiy `Trainer.train()` Kaggle'da GPU
bilan bajariladi — bu yerda konfiguratsiyani tayyorlaymiz va tekshiramiz; path'dan mustaqil.)

In [ ]:
# L13 [K4] giperparametrlari (Kaggle'da TrainingArguments(**hparams) ga beriladi)
hparams = dict(learning_rate=2e-5, per_device_train_batch_size=16,
               num_train_epochs=3, warmup_steps=100)
print("Giperparametrlar:", hparams)
# Kaggle (USE_TRANSFORMERS=True) da haqiqiy fine-tuning:
#   from transformers import (AutoModelForSequenceClassification,
#                             TrainingArguments, Trainer)
#   args = TrainingArguments(output_dir="bert_out", report_to=[], **hparams)
#   model = AutoModelForSequenceClassification.from_pretrained(
#       "distilbert-base-multilingual-cased", num_labels=2)
#   Trainer(model=model, args=args, train_dataset=ds).train()

In [ ]:
assert hparams["learning_rate"] == 2e-5
assert hparams["per_device_train_batch_size"] == 16
assert hparams["num_train_epochs"] == 3
assert hparams["warmup_steps"] == 100
print("To'g'ri! Giperparametrlar L13 [K4] bilan mos.")

### 4C. Birgalikda: m13 ni o'qitish va bashorat

`m13 FineTunedClassifier` ni korpusda o'qiting (mahalliy: LogReg fallback) va bir sharhni tasniflang.

In [ ]:
sent = FineTunedClassifier()
sent.fit(sharhlar, yorliqlar, epochs=3, batch_size=16, lr=2e-5)
namuna = "mahsulot juda sifatli va arzon"
bashorat = sent.predict(namuna)
print(repr(namuna), "->", bashorat)

In [ ]:
assert bashorat in ("ijobiy", "salbiy")
print("To'g'ri! predict ijobiy/salbiy qaytardi.")

### 4D. Mustaqil (siz qilasiz): ehtimolliklar

Salbiy sharh uchun `predict_proba` ni chaqiring va natijani tekshiring. (Kichik data — aniqlik
demo-sifatli, bu halol.)

In [ ]:
proba = sent.predict_proba("yetkazib berish juda sekin va yomon")
print("Ehtimolliklar:", {k: round(v, 3) for k, v in proba.items()})

assert set(proba.keys()) == {"ijobiy", "salbiy"}
assert all(0.0 <= v <= 1.0 for v in proba.values())
assert abs(sum(proba.values()) - 1.0) < 1e-6
print("To'g'ri! predict_proba 2 kalitli ({ijobiy, salbiy}) dict, yig'indi = 1.")

## 5. Loyihaga ulash — `m13 FineTunedClassifier`

Bugungi kod `capstone/modules/m13_bert_classifier.py` ga yig'ilgan (interfeys `contracts.py` dan).
U **haqiqiy modul**: `save`/`load` bor va m15 agent (`sentiment_classify`) hamda `app.py` (FastAPI, M4)
uni ishlatadi.

In [ ]:
import tempfile
api = FineTunedClassifier()
for m in ("fit", "predict", "predict_proba", "save", "load"):
    assert hasattr(api, m), "yetishmayotgan metod: " + m

api.fit(sharhlar, yorliqlar)
_p = os.path.join(tempfile.gettempdir(), "m13_test.pkl")
api.save(_p)
api2 = FineTunedClassifier()
api2.load(_p)
qayta = api2.predict("mahsulot zo'r va sifatli")
print("Yuklangan modeldan bashorat:", qayta)
assert qayta in ("ijobiy", "salbiy")
os.remove(_p)
print("To'g'ri! m13 shartnomaga mos; save/load ishlaydi.")

**Git (o'z repozitoriyangizda):**
```bash
git add capstone/modules/m13_bert_classifier.py
git commit -m "P13: m13 FineTunedClassifier qo'shildi"
git push
```

## 6. Tadqiqot savoli + yakun

**Tajriba:** Kaggle'da `USE_TRANSFORMERS=True` qilib DistilBERT'ni fine-tune qiling va F1 ni offline
LogReg fallback bilan solishtiring. Quyida fallback aniqligini ko'ramiz.

In [ ]:
# offline fallback aniqligi (o'qitilgan korpusning o'zida — faqat namoyish)
togri = sum(1 for t, y in zip(sharhlar, yorliqlar) if sent.predict(t) == y)
print("Fallback (LogReg) aniqligi (train):", round(togri / len(sharhlar), 3))
print("Eslatma: bu o'quv korpusda; haqiqiy baho alohida test to'plamda olinadi.")

**O'ylab ko'ring:** mBERT WordPiece 'o'rganaman' ni ~5 bo'lakka kesadi (L13 [M]). Bu sentiment
klassifikatsiyaga qanday ta'sir qiladi? XLM-RoBERTa nega o'zbek uchun yaxshiroq bo'lishi mumkin?
Fine-tuning noldan o'qitishdan nimasi bilan arzonroq?

---
**Chiqish chiptasi:** Bugun eng tushunarsiz qolgan narsa nima? (Bir jumla.)